# Let's clearly define a lag-phase analysis:

Determine the significance of a selected year of vaccine coverage and how the following disease cases change over x number of years.

Reworded AI: Determine whether vaccine coverage in a selected year is significantly associated with changes in disease incidence over the following x years.

In [18]:
import pandas as pd

df = pd.read_csv("../app/data/cases_coverage_ms.csv")
df.head()

,year,state,measles_cases,mumps_cases,pertussis_cases,measles_coverage_pct,measles_n,measles_n_vaccinated,mumps_coverage_pct,mumps_n,mumps_n_vaccinated,pertussis_coverage_pct,pertussis_n,pertussis_n_vaccinated,population,measles_cases_per_100k,mumps_cases_per_100k,pertussis_cases_per_100k
0,1995,AK,NaN,13.0,1.0,89.86,194.0,177.0,89.86,194.0,177.0,77.58,194.0,159.0,604412.0,NaN,2.150851,0.165450
1,1996,AK,63.0,3.0,4.0,84.55,260.0,225.0,84.55,260.0,225.0,78.25,260.0,210.0,608569.0,10.352154,0.492960,0.657280
2,1997,AK,48.0,4.0,14.0,87.41,291.0,257.0,87.41,291.0,257.0,80.98,291.0,242.0,612968.0,7.830751,0.652563,2.283969
3,1998,AK,29.0,2.0,15.0,87.06,34.0,30.0,87.06,34.0,30.0,82.01,34.0,28.0,619932.0,4.677932,0.322616,2.419620
4,1999,AK,29.0,3.0,5.0,90.67,349.0,321.0,90.67,349.0,321.0,83.54,349.0,296.0,624779.0,4.641641,0.480170,0.800283


In [19]:
import pandas as pd
import numpy as np
from scipy.stats import pearsonr


def pearson_lag_analysis(df, state, disease, start_year, max_lag=5):

    coverage_col = f"{disease}_coverage_pct"
    incidence_col = f"{disease}_cases_per_100k"

    # Filter selected state
    data = df.loc[
        (df["state"] == state) &
        (df["year"] >= start_year),
        ["year", coverage_col, incidence_col]
    ].copy()

    data = data.sort_values("year")

    # Ensure one observation per year
    if data["year"].duplicated().any():
        raise ValueError("Duplicate years found for selected state.")

    data = data.set_index("year")

    # Identify coverage years with valid observations
    # at every lag from 1 through max_lag
    valid_years = []

    for year in data.index:

        coverage = data.at[year, coverage_col]

        if not np.isfinite(coverage):
            continue

        valid = True

        for lag in range(1, max_lag + 1):

            future_year = year + lag

            if future_year not in data.index:
                valid = False
                break

            incidence = data.at[future_year, incidence_col]

            if not np.isfinite(incidence):
                valid = False
                break

        if valid:
            valid_years.append(year)

    results = []

    # Calculate Pearson correlation for each lag
    for lag in range(1, max_lag + 1):

        coverage_values = []
        incidence_values = []

        for year in valid_years:

            coverage_values.append(
                data.at[year, coverage_col]
            )

            incidence_values.append(
                data.at[year + lag, incidence_col]
            )

        n = len(coverage_values)

        if (
            n >= 3
            and len(set(coverage_values)) > 1
            and len(set(incidence_values)) > 1
        ):
            r, p = pearsonr(
                coverage_values,
                incidence_values
            )
        else:
            r, p = np.nan, np.nan

        results.append({
            "state": state,
            "disease": disease,
            "start_year": start_year,
            "lag_years": lag,
            "n": n,
            "pearson_r": r,
            "p_value": p,
            "significant_0_05": (
                p < 0.05 if pd.notna(p) else False
            )
        })

    return pd.DataFrame(results)

In [23]:
results = pearson_lag_analysis(
    df=df,
    state="CA",
    disease="mumps",
    start_year=1995,
    max_lag=10
)

results

,state,disease,start_year,lag_years,n,pearson_r,p_value,significant_0_05
0,CA,mumps,1995,1,13,-0.555829,0.048569,True
1,CA,mumps,1995,2,13,-0.715150,0.005999,True
2,CA,mumps,1995,3,13,-0.651247,0.015902,True
3,CA,mumps,1995,4,13,-0.422100,0.150791,False
4,CA,mumps,1995,5,13,-0.690482,0.008984,True
5,CA,mumps,1995,6,13,-0.679646,0.010604,True
6,CA,mumps,1995,7,13,-0.678382,0.010807,True
7,CA,mumps,1995,8,13,-0.566469,0.043549,True
8,CA,mumps,1995,9,13,0.077741,0.800704,False
9,CA,mumps,1995,10,13,0.268265,0.375516,False


### Time-lagged Pearson correlation analysis was conducted to evaluate associations between annual vaccination coverage and subsequent disease incidence at different lag intervals. A common observation window was used across all lag intervals to maintain consistent sample sizes and vaccination coverage years, allowing comparisons of correlation strength across time delays.

In [21]:
import plotly.graph_objects as go


def graph_pearson_lag_results(results):

    df_plot = results.sort_values("lag_years").copy()

    fig = go.Figure()

    fig.add_trace(
        go.Scatter(
            x=df_plot["lag_years"],
            y=df_plot["pearson_r"],
            mode="lines+markers",
            name="Pearson r",
            line=dict(color="#3498DB", width=2),
            marker=dict(
                size=11,
                color=[
                    "#2ECC71" if significant else "#3498DB"
                    for significant in df_plot["significant_0_05"]
                ],
                line=dict(color="white", width=1)
            ),
            customdata=df_plot[
                ["n", "p_value"]
            ].to_numpy(),
            hovertemplate=(
                "Lag: %{x} years<br>"
                "Pearson r: %{y:.3f}<br>"
                "n: %{customdata[0]}<br>"
                "p-value: %{customdata[1]:.4f}"
                "<extra></extra>"
            )
        )
    )

    fig.add_hline(
        y=0,
        line_dash="dash",
        line_color="gray"
    )

    state = df_plot["state"].iloc[0]
    disease = df_plot["disease"].iloc[0].capitalize()
    start_year = df_plot["start_year"].iloc[0]

    fig.update_layout(
        title=(
            f"{state} — {disease} Vaccination Coverage "
            f"vs. Future Incidence"
            f"<br><sup>Starting year: {start_year}</sup>"
        ),
        xaxis_title="Lag (Years)",
        yaxis_title="Pearson Correlation (r)",
        xaxis=dict(
            tickmode="linear",
            dtick=1
        ),
        yaxis=dict(
            range=[-1, 1],
            zeroline=False
        ),
        template="plotly_white",
        height=500,
        hovermode="closest"
    )

    return fig

In [22]:
fig = graph_pearson_lag_results(results)
fig.show()

Why significance doesn't necessarily mean a vaccine effect
There are three important considerations for your capstone:
1. Multiple comparisons: You tested 10 different lag durations. Some significant results can arise by chance. Adjusting p-values helps address this.
2. Time-series dependence: Vaccination coverage and disease incidence can follow long-term trends. The annual observations may not be statistically independent, making conventional Pearson p-values less reliable.
3. Population differences: Your vaccination coverage data concern young children, while reported mumps incidence includes other age groups. These are population-level associations, not individual-level vaccine outcomes.

```
California demonstrated negative correlations between childhood mumps vaccination coverage and subsequent mumps incidence across several lag durations. The strongest association occurred at a two-year lag (r = −0.715, n = 13, unadjusted p = 0.006). Although statistically significant under the conventional Pearson test, these findings are exploratory and do not establish a causal relationship.
```